# Merge shard CoT outputs (Qwen2.5-Math)

Gabung hasil generate per-shard (disjoint soal) jadi satu file per dataset, ditaruh di `data/cot/<folder>/` dengan **nama file sama** kayak output Kaggle.

Sumber (download dari Kaggle):
- **numglue**: `cot_sft_outputs_SHARD0/cot/numglue/` + `cot_sft_outputs_SHARD1/cot/numglue/`
- **easy**:    `cot_sft_outputs_SHARD0_easy/cot/easy/` + `cot_sft_outputs_SHARD1_easy/cot/easy/`

Tiap folder berisi:
- `cot_<ds>_<teacher>.jsonl` (kandidat, JSONL)
- `correct_<ds>_<teacher>.jsonl` (lolos judge, JSONL)
- `correct_<ds>_<teacher>.jsonl.progress` (checkpoint judge, **TSV** `id\tcandidate_idx`)

Ketiganya digabung (concat). Shard disjoint -> tetap di-dedup buat jaga-jaga: JSONL by `(id, candidate_idx)`, progress by baris penuh.

Tujuan:
- numglue -> `data/cot/num_glue/` (samaan sama file DeepSeek yang udah ada)
- easy    -> `data/cot/easy/`

Jalankan notebook ini **dari root repo** (`FP_NLP/`).

In [ ]:
from pathlib import Path
import json

# ROOT = root repo. Notebook ada di notebooks/, jadi naik 1 level. Ganti kalau jalan dari tempat lain.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'data').exists(), f'jalankan dari root repo; ROOT sekarang: {ROOT}'

TEACHER_TAG = 'Qwen2.5-Math-7B-Instruct'   # masuk nama file: cot_<ds>_<TEACHER_TAG>.jsonl

# dataset -> {ds di nama file, folder target, list folder shard sumber}
JOBS = {
    'numglue': {
        'ds':     'numglue',
        'target': ROOT / 'data' / 'cot' / 'num_glue',
        'shards': [
            ROOT / 'cot_sft_outputs_SHARD0' / 'cot' / 'numglue',
            ROOT / 'cot_sft_outputs_SHARD1' / 'cot' / 'numglue',
        ],
    },
    'easy': {
        'ds':     'easy',
        'target': ROOT / 'data' / 'cot' / 'easy',
        'shards': [
            ROOT / 'cot_sft_outputs_SHARD0_easy' / 'cot' / 'easy',
            ROOT / 'cot_sft_outputs_SHARD1_easy' / 'cot' / 'easy',
        ],
    },
}

print('ROOT:', ROOT)
for name, job in JOBS.items():
    print(f"{name}: {len(job['shards'])} shard -> {job['target']}")

In [ ]:
def merge_jsonl(srcs, dst, dedup_key=('id', 'candidate_idx')):
    """Concat JSONL -> dst, dedup by dedup_key. Return (ditulis, dilewati)."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    seen, lines, skipped = set(), [], 0
    for s in srcs:
        if not s.exists():
            print('   (TIDAK ADA, dilewati):', s)
            continue
        with open(s, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                if dedup_key:
                    d = json.loads(line)
                    k = tuple(d.get(x) for x in dedup_key)
                    if k in seen:
                        skipped += 1
                        continue
                    seen.add(k)
                lines.append(line)
    with open(dst, 'w', encoding='utf-8') as o:
        for ln in lines:
            o.write(ln + '\n')
    return len(lines), skipped


def merge_text(srcs, dst):
    """Concat file teks (mis. .progress TSV) -> dst, dedup by baris penuh. Return (ditulis, dilewati)."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    seen, lines, skipped = set(), [], 0
    for s in srcs:
        if not s.exists():
            print('   (TIDAK ADA, dilewati):', s)
            continue
        with open(s, encoding='utf-8') as f:
            for line in f:
                line = line.rstrip('\n')
                if not line:
                    continue
                if line in seen:
                    skipped += 1
                    continue
                seen.add(line)
                lines.append(line)
    with open(dst, 'w', encoding='utf-8') as o:
        for ln in lines:
            o.write(ln + '\n')
    return len(lines), skipped


for name, job in JOBS.items():
    ds, tgt = job['ds'], job['target']
    print('=' * 12, name, '->', tgt)
    # JSONL: cot (kandidat) + correct (lolos judge)
    for kind in ('cot', 'correct'):
        fname = f'{kind}_{ds}_{TEACHER_TAG}.jsonl'
        srcs = [sh / fname for sh in job['shards']]
        written, skipped = merge_jsonl(srcs, tgt / fname)
        print(f'  {kind:8s}: {written} baris (dup dilewati: {skipped}) -> {tgt / fname}')
    # TSV checkpoint: correct_*.jsonl.progress
    pname = f'correct_{ds}_{TEACHER_TAG}.jsonl.progress'
    srcs = [sh / pname for sh in job['shards']]
    written, skipped = merge_text(srcs, tgt / pname)
    print(f'  progress: {written} baris (dup dilewati: {skipped}) -> {tgt / pname}')

In [ ]:
# Verifikasi hasil merge.
for name, job in JOBS.items():
    tgt, ds = job['target'], job['ds']
    for kind in ('cot', 'correct'):
        p = tgt / f'{kind}_{ds}_{TEACHER_TAG}.jsonl'
        if not p.exists():
            print('MISSING:', p); continue
        rows = [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
        soal = {r.get('id') for r in rows}
        print(f'{name}/{kind}: {len(rows)} baris | {len(soal)} soal unik | {p}')
    pp = tgt / f'correct_{ds}_{TEACHER_TAG}.jsonl.progress'
    if pp.exists():
        n = sum(1 for l in open(pp, encoding='utf-8') if l.strip())
        print(f'{name}/progress: {n} baris | {pp}')